In [33]:
import pyodbc
import pandas as pd
from sqlalchemy import create_engine
import urllib

# Connection string
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "Server=LAMOSKI;"
    "Database=AirBnB_London;"
    "Trusted_Connection=yes;"
    "Encrypt=no;"
)


engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)


conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=LAMOSKI;"
    "Database=AirBnB_London;"
    "Trusted_Connection=yes;"
    "Encrypt=no;"
)

cursor = conn.cursor()
cursor.execute("SELECT @@VERSION")
print(cursor.fetchone())
df_Bronze = pd.read_csv(r"C:\Users\olami\OneDrive\Desktop\AirBnB\Data\listings.csv\listings.csv")
df_Bronze.to_sql('Listings_London', con=engine, schema='Bronze', if_exists='replace', index=False)

('Microsoft SQL Server 2022 (RTM) - 16.0.1000.6 (X64) \n\tOct  8 2022 05:58:25 \n\tCopyright (C) 2022 Microsoft Corporation\n\tDeveloper Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\n',)


C:\Users\olami\AppData\Local\Temp\ipykernel_44832\1941566332.py:33: DtypeWarning: Columns (0: source) have mixed types. Specify dtype option on import or set low_memory=False.
  df_Bronze = pd.read_csv(r"C:\Users\olami\OneDrive\Desktop\AirBnB\Data\listings.csv\listings.csv")


-1

In [34]:
# Checking the shape of the DataFrame after loading it into Bronze schema
df_Bronze = pd.read_csv(r"C:\Users\olami\OneDrive\Desktop\AirBnB\Data\listings.csv\listings.csv")
print("Observations:", df_Bronze.shape[0])
print("Variables:", df_Bronze.shape[1])
print(df_Bronze.head())



C:\Users\olami\AppData\Local\Temp\ipykernel_44832\4291969000.py:2: DtypeWarning: Columns (0: source) have mixed types. Specify dtype option on import or set low_memory=False.
  df_Bronze = pd.read_csv(r"C:\Users\olami\OneDrive\Desktop\AirBnB\Data\listings.csv\listings.csv")


Observations: 91954
Variables: 90
      id                         listing_url       scrape_id last_scraped  \
0  11551  https://www.airbnb.com/rooms/11551  20260917015706   2026-09-19   
1  13913  https://www.airbnb.com/rooms/13913  20260917015706   2026-09-19   
2  15400  https://www.airbnb.com/rooms/15400  20260917015706   2026-09-19   
3  24328  https://www.airbnb.com/rooms/24328  20260917015706   2026-09-19   
4  25081  https://www.airbnb.com/rooms/25081  20260917015706   2026-09-21   

            source                                               name  \
0              NaN                       Artistic London Pied-à-Terre   
1              NaN                Holiday London DB Room Let-on going   
2              NaN                Bright Chelsea  Apartment. Chelsea!   
3              NaN     Battersea live/work artist house with car park   
4  previous scrape  Lovely bright flat for 1-4p 20 m to Covent Garden   

                                         description  neighborho

In [35]:
# Selecting the columns needed for the Silver schema
columns_needed = [
    "id",
    "name",
    "listing_url",
    "host_name",
    "host_is_superhost",
  
    "host_listings_count",
    "host_identity_verified",
    "neighbourhood_cleansed",
    
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",

    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities",
    'price',
   
    "price_quote_raw",
    "minimum_nights",
    "calendar_last_scraped",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_location",
  
]



In [36]:
# Creating the Silver schema DataFrame and generating a missing values report
df_Silver = df_Bronze[columns_needed]
missing_report = pd.DataFrame({
    "missing_count": df_Silver.isna().sum(),
    "missing_pct": (100 * df_Silver.isna().mean()).round(2)
})
print(missing_report.sort_values("missing_pct", ascending=False))

                        missing_count  missing_pct
price                           32049        34.85
price_quote_raw                 29225        31.78
review_scores_location          20734        22.55
review_scores_rating            20694        22.50
beds                             1990         2.16
minimum_nights                    309         0.34
bathrooms_text                    108         0.12
bedrooms                           45         0.05
neighbourhood_cleansed              0         0.00
host_identity_verified              0         0.00
host_listings_count                 0         0.00
host_is_superhost                   0         0.00
host_name                           0         0.00
listing_url                         0         0.00
name                                0         0.00
id                                  0         0.00
accommodates                        0         0.00
longitude                           0         0.00
property_type                  

### Handling Missing Price Values

The `price` column is the target variable for this project. A total of 32,049 listings (34.85%) had missing prices. These records were removed because they could not be used to train the prediction model, leaving 59,905 listings for further analysis.

In [37]:
# dropping rows with missing price values to create the Silver model DataFrame
df_silver_model = df_Silver.dropna(subset=["price"]).copy()

# Check results
print("Original listings:", len(df_Silver))
print("Listings retained:", len(df_silver_model))
print("Listings removed:", len(df_Silver) - len(df_silver_model))

Original listings: 91954
Listings retained: 59905
Listings removed: 32049


In [38]:
# Remove listings with missing prices
df_silver_model = df_Silver.dropna(subset=["price"]).copy()

# Check missing values
missing_report = pd.DataFrame({
    "Missing_Count": df_silver_model.isna().sum(),
    "Missing_Percentage": (
        df_silver_model.isna().mean() * 100
    ).round(2)
})

# Display only columns with missing values
missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values("Missing_Count", ascending=False)

print("Total listings:", len(df_silver_model))
print(missing_report.to_string())

Total listings: 59905
                        Missing_Count  Missing_Percentage
review_scores_location          12263               20.47
review_scores_rating            12258               20.46
beds                             1502                2.51
minimum_nights                    298                0.50
bathrooms_text                     92                0.15
bedrooms                           24                0.04


### Handling Missing Bed Values

The `beds` column contained 1,502 missing values (2.51%). Further analysis was carried out by grouping listings according to `room_type` and guest capacity (`accommodates`).

The results showed that the median number of beds generally increased with guest capacity. For example, entire homes accommodating 2 guests had a median of 1 bed, while those accommodating 6 and 10 guests had medians of 3 and 5 beds, respectively.

**Decision:** Missing bed values were filled using the median for each combination of `room_type` and `accommodates`. This method was chosen because it considers the size of the accommodation and reduces the influence of unusually high bed counts. All 59,905 listings were retained.

In [39]:

beds_capacity_report = (
    df_silver_model
    .groupby(["room_type", "accommodates"])
    .agg(
        total_listings=("beds", "size"),
        missing_beds=("beds", lambda x: x.isna().sum()),
        mean_beds=("beds", "mean"),
        median_beds=("beds", "median"),
        percentile_75=("beds", lambda x: x.quantile(0.75)),
        min_beds=("beds", "min"),
        max_beds=("beds", "max")
    )
    .reset_index()
)

beds_capacity_report["missing_pct"] = (
    beds_capacity_report["missing_beds"]
    / beds_capacity_report["total_listings"] * 100
).round(2)

print(
    beds_capacity_report
    .sort_values(["room_type", "accommodates"])
    .round(2)
    .to_string(index=False)
)


      room_type  accommodates  total_listings  missing_beds  mean_beds  median_beds  percentile_75  min_beds  max_beds  missing_pct
Entire home/apt             1             463            11       1.07          1.0           1.00       1.0       4.0         2.38
Entire home/apt             2           10547           229       1.09          1.0           1.00       1.0      12.0         2.17
Entire home/apt             3            3518            62       1.70          2.0           2.00       1.0       4.0         1.76
Entire home/apt             4           11452           227       2.08          2.0           2.00       1.0       9.0         1.98
Entire home/apt             5            3546            53       2.92          3.0           3.00       1.0       7.0         1.49
Entire home/apt             6            5916           138       3.20          3.0           4.00       1.0       9.0         2.33
Entire home/apt             7            1244            23       4.13      

In [40]:
import numpy as np

df_Silver_beds_clean = df_silver_model.copy()

# Preserve the original bed counts
df_Silver_beds_clean["beds_original"] = df_Silver_beds_clean["beds"]

# Flag missing beds
df_Silver_beds_clean["beds_missing"] = (
    df_Silver_beds_clean["beds"].isna().astype(int)
)

# Median by room type and guest capacity
group_median = (
    df_Silver_beds_clean
    .groupby(["room_type", "accommodates"])["beds"]
    .transform("median")
)

# Fallback: room-type median
room_median = (
    df_Silver_beds_clean
    .groupby("room_type")["beds"]
    .transform("median")
)

# Overall median
overall_median = df_Silver_beds_clean["beds"].median()

# Fill missing values
df_Silver_beds_clean["beds"] = (
    df_Silver_beds_clean["beds"]
    .fillna(group_median)
    .fillna(room_median)
    .fillna(overall_median)
)

# Round only imputed values to whole beds
mask = df_Silver_beds_clean["beds_missing"] == 1

df_Silver_beds_clean.loc[mask, "beds"] = np.floor(
    df_Silver_beds_clean.loc[mask, "beds"] + 0.5
)

print("Missing before:", df_Silver_beds_clean["beds_missing"].sum())
print("Missing after:", df_Silver_beds_clean["beds"].isna().sum())

Missing before: 1502
Missing after: 0


### Handling Missing Minimum Nights Values

The `minimum_nights` column contained 298 missing values (0.50%). Further analysis showed that 297 missing values belonged to hotel rooms, while only one belonged to an entire rental unit.

The mean minimum stay was 3.74 nights, compared with a median of 2 nights, indicating a right-skewed distribution.

**Decision:** Missing values were filled using the median minimum stay based on `room_type`. This method was chosen to reduce the influence of unusually long minimum stays while retaining all 59,905 listings.

In [41]:
nights_report = (
    df_silver_model
    .groupby("room_type")
    .agg(
        total_listings=("minimum_nights", "size"),
        missing_nights=("minimum_nights", lambda x: x.isna().sum()),
        mean_nights=("minimum_nights", "mean"),
        median_nights=("minimum_nights", "median"),
        percentile_75=("minimum_nights", lambda x: x.quantile(0.75)),
        min_nights=("minimum_nights", "min"),
        max_nights=("minimum_nights", "max")
    )
)

# Calculate missing percentage
nights_report["missing_pct"] = (
    nights_report["missing_nights"]
    / nights_report["total_listings"] * 100
).round(2)

print(
    nights_report
    .sort_values("missing_nights", ascending=False)
    .round(2)
    .to_string()
)

                 total_listings  missing_nights  mean_nights  median_nights  percentile_75  min_nights  max_nights  missing_pct
room_type                                                                                                                      
Private room              19085             297         2.97            1.0            2.0         1.0       365.0         1.56
Entire home/apt           40521               1         4.11            2.0            3.0         1.0       365.0         0.00
Hotel room                  134               0         1.12            1.0            1.0         1.0         3.0         0.00
Shared room                 165               0         1.45            1.0            1.0         1.0        30.0         0.00


In [42]:
# Create a copy of the dataset
df_nights_clean = df_silver_model.copy()

# Flag missing values
df_nights_clean["minimum_nights_missing"] = (
    df_nights_clean["minimum_nights"].isna().astype(int)
)

# Calculate median by room type
room_median = (
    df_nights_clean
    .groupby("room_type")["minimum_nights"]
    .transform("median")
)

# Fill missing values using room-type median
df_nights_clean["minimum_nights"] = (
    df_nights_clean["minimum_nights"]
    .fillna(room_median)
    .fillna(df_nights_clean["minimum_nights"].median())
)

# Verify
print("Missing before:",
      df_nights_clean["minimum_nights_missing"].sum())

print("Missing after:",
      df_nights_clean["minimum_nights"].isna().sum())

Missing before: 298
Missing after: 0


### Handling Missing Bathroom Values

The extracted `bathrooms_count` column contained 98 missing values (0.16%). Analysis by `room_type` and guest capacity showed that the median bathroom count was 1 for all groups with missing values and available observations.

**Decision:** Missing bathroom counts were assigned the group median, with the room-type median used as a fallback. Unusually high bathroom counts were flagged separately for investigation, and bathroom access was retained as an additional feature.

In [43]:

#Create the bathroom DataFrame
df_bathroom = df_silver_model.copy()

#Extract numerical bathroom counts
df_bathroom["bathrooms_count"] = pd.to_numeric(
    df_bathroom["bathrooms_text"]
    .str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Handle half-bath descriptions
half_bath = df_bathroom["bathrooms_text"].str.contains(
    "half.bath", case=False, na=False, regex=True
)

df_bathroom.loc[half_bath, "bathrooms_count"] = 0.5

# Step 3: Investigate bathrooms by room type and guest capacity
bathroom_capacity_report = (
    df_bathroom
    .groupby(["room_type", "accommodates"])
    .agg(
        total_listings=("bathrooms_count", "size"),
        missing_bathrooms=("bathrooms_count", lambda x: x.isna().sum()),
        mean_bathrooms=("bathrooms_count", "mean"),
        median_bathrooms=("bathrooms_count", "median"),
        percentile_75=("bathrooms_count", lambda x: x.quantile(0.75)),
        min_bathrooms=("bathrooms_count", "min"),
        max_bathrooms=("bathrooms_count", "max")
    )
    .reset_index()
)

# Step 4: Calculate missing percentage
bathroom_capacity_report["missing_pct"] = (
    bathroom_capacity_report["missing_bathrooms"]
    / bathroom_capacity_report["total_listings"] * 100
).round(2)

# Step 5: Display results
print(
    bathroom_capacity_report
    .sort_values(["room_type", "accommodates"])
    .round(2)
    .to_string(index=False)
)

      room_type  accommodates  total_listings  missing_bathrooms  mean_bathrooms  median_bathrooms  percentile_75  min_bathrooms  max_bathrooms  missing_pct
Entire home/apt             1             463                  0            1.03              1.00           1.00            0.0            3.0         0.00
Entire home/apt             2           10547                  0            1.03              1.00           1.00            0.0            6.0         0.00
Entire home/apt             3            3518                  0            1.06              1.00           1.00            0.0            3.0         0.00
Entire home/apt             4           11452                  3            1.31              1.00           2.00            0.0            5.5         0.03
Entire home/apt             5            3546                  0            1.50              1.00           2.00            0.0            6.0         0.00
Entire home/apt             6            5916             

In [44]:

df_bathroom_clean = df_bathroom.copy()

# Preserve original numerical bathroom counts
df_bathroom_clean["bathrooms_original"] = (
    df_bathroom_clean["bathrooms_count"]
)

# Flag missing bathroom counts
df_bathroom_clean["bathrooms_missing"] = (
    df_bathroom_clean["bathrooms_count"].isna().astype(int)
)

# Median by room type and guest capacity
group_median = (
    df_bathroom_clean
    .groupby(["room_type", "accommodates"])["bathrooms_count"]
    .transform("median")
)

# Fallback median by room type
room_median = (
    df_bathroom_clean
    .groupby("room_type")["bathrooms_count"]
    .transform("median")
)

# Overall fallback median
overall_median = df_bathroom_clean["bathrooms_count"].median()

# Fill missing bathroom counts
df_bathroom_clean["bathrooms_count"] = (
    df_bathroom_clean["bathrooms_count"]
    .fillna(group_median)
    .fillna(room_median)
    .fillna(overall_median)
)

# Verify
print("Missing before:",
      df_bathroom_clean["bathrooms_missing"].sum())

print("Missing after:",
      df_bathroom_clean["bathrooms_count"].isna().sum())

# Check assigned values
print(
    df_bathroom_clean.loc[
        df_bathroom_clean["bathrooms_missing"] == 1,
        "bathrooms_count"
    ].value_counts()
)


Missing before: 98
Missing after: 0
bathrooms_count
1.0    98
Name: count, dtype: int64


### Handling Missing Bedroom Values

The `bedrooms` column contained 24 missing values (0.04%). All 24 listings were manually investigated using their Airbnb listing pages. The investigation confirmed that 23 listings had no separate bedrooms, while one listing had 1 bedroom.

**Decision:** The missing values were manually corrected using the verified bedroom counts, retaining all listings for further analysis.

In [45]:
# Create a copy of the dataset
df_bedrooms_clean = df_silver_model.copy()

# Get the original 24 missing-bedroom listing URLs
missing_urls = missing_bedrooms["listing_url"].tolist()

# Confirm the number of listings
assert len(missing_urls) == 24

# Set all 24 missing bedroom values to 0
df_bedrooms_clean.loc[
    df_bedrooms_clean["listing_url"].isin(missing_urls),
    "bedrooms"
] = 0

# Set listing number 3 to 1 bedroom
df_bedrooms_clean.loc[
    df_bedrooms_clean["listing_url"] == missing_urls[2],
    "bedrooms"
] = 1

# Verify
print("Remaining missing bedrooms:",
      df_bedrooms_clean["bedrooms"].isna().sum())

print(
    df_bedrooms_clean.loc[
        df_bedrooms_clean["listing_url"].isin(missing_urls),
        ["listing_url", "property_type", "bedrooms"]
    ].to_string(index=False)
)

Remaining missing bedrooms: 0
                                     listing_url             property_type  bedrooms
            https://www.airbnb.com/rooms/6919139                 Camper/RV       0.0
           https://www.airbnb.com/rooms/12004105              Private room       0.0
           https://www.airbnb.com/rooms/22241379              Entire place       1.0
           https://www.airbnb.com/rooms/34203528                       Hut       0.0
           https://www.airbnb.com/rooms/33573140                 Houseboat       0.0
           https://www.airbnb.com/rooms/33587684                 Camper/RV       0.0
           https://www.airbnb.com/rooms/39880406 Private room in tiny home       0.0
           https://www.airbnb.com/rooms/43419503                      Dome       0.0
           https://www.airbnb.com/rooms/48583067                      Tent       0.0
           https://www.airbnb.com/rooms/49391946              Entire place       0.0
           https://www.airbnb.com/r